# Metodologia de forecast historico de demanda SOLIDO

Este notebook documenta el modulo predictivo vigente. El forecast se mantiene
separado del descriptivo y de clusters: usa directamente la base historica
acumulada, la limpia con la misma regla operativa y modela exclusivamente pedidos
`SOLIDO` confirmados.

El ejecutor oficial es `run_forecast_solidos.py`.

## 1. Alcance modelado y exclusiones

```text
bases de datos historicas/historic_sales_acum.csv (receta/estructura completa)
          |
          | limpiar + clasificar receta + conservar SOLIDO real
          v
cliente + mercado + pais + producto + color + semana
          |
          v
modelos de demanda y forecast futuro
```

**Objetivo pronosticado:** tallos confirmados semanales.

**Incluye:** pedidos `SOLIDO` confirmados cuando la fuente informa estado. En
el acumulado actual de ventas, que no contiene columna `ESTADO`, todas las
lineas se interpretan como historia comercial observada.

**Excluye:** `SURTIDO`, `SURTIDO_M`, `RAINBOW`, `BOUQUET`, `BQT`, `COMBO` y
otras estructuras mixtas. Los campos originales de la base acumulada permiten
identificar estas categorias antes de filtrar el universo modelado. Aunque
una fila de `BQT` o `COMBO` contenga el texto `Solido`, no se considera solido
pronosticable por color.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

OUTPUT_DIR = ROOT / "resultados" / "forecast_solidos"

def read_forecast_output(name, parse_dates=None):
    path = OUTPUT_DIR / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"No existe {path}. Ejecuta primero run_forecast_solidos.py.")
    return pd.read_csv(path, parse_dates=parse_dates, low_memory=False)

evaluation = read_forecast_output("solid_forecast_model_evaluation")
future = read_forecast_output("solid_forecast_future", parse_dates=["week_start"])
test = read_forecast_output("solid_forecast_test_predictions", parse_dates=["week_start"])

## 2. Modelos evaluados

El modulo compara alternativas con validacion temporal: las semanas finales se
reservan como test, evitando usar informacion futura para entrenar.

| Modelo | Logica | Uso como referencia |
|---|---|---|
| Baseline reciente | Mediana de semanas recientes | Punto de partida simple |
| Estacional anual | Comportamiento de semanas equivalentes de anos previos | Captura repeticion anual |
| Boosting | Aprende ocurrencia y volumen con predictores historicos/operativos | Captura interacciones y comportamiento reciente |

El boosting opera en dos etapas:

1. `probabilidad_compra`: probabilidad de que el cliente compre ese
   producto-color en la semana.
2. `volumen_si_compra`: tallos esperados cuando la compra ocurre. Se entrena
   en escala logaritmica y se retransforma con una correccion de sesgo aprendida
   solamente en entrenamiento, para reducir subpronostico de volumen en picos.

La seleccion final se basa en rendimiento sobre test y se presenta en el
dashboard junto con sus metricas.

In [ ]:
evaluation.sort_values("WAPE")[
    [c for c in ["modelo", "WAPE", "RMSE", "bias_pct", "modelo_seleccionado"] if c in evaluation.columns]
]

## 3. Metricas y lectura comercial

- **WAPE:** error total ponderado por volumen. Es la metrica principal para saber
  cuanta demanda total se asigna incorrectamente.
- **RMSE:** penaliza errores grandes, util para detectar clientes/productos con
  desviaciones severas.
- **Bias:** indica si el modelo tiende a predecir por debajo o por encima.

El WAPE puede ser mayor al mirar cliente-producto-color que al mirar un mercado
total: el modelo puede aproximar el volumen global, pero equivocarse en la
distribucion fina. Ambos niveles deben revisarse antes de tomar decisiones.

In [ ]:
used = evaluation.loc[evaluation["modelo_seleccionado"].astype(bool), "modelo"].iloc[0]
used_test = test[test["modelo"].eq(used)].copy() if "modelo" in test.columns else test.copy()

def wape(frame, group_cols):
    summary = frame.groupby(group_cols, as_index=False).agg(
        real=("tallos", "sum"),
        pred=("prediccion", "sum"),
    )
    return (summary["real"] - summary["pred"]).abs().sum() / summary["real"].abs().sum()

print("Modelo usado:", used)
for label, keys in {
    "Detalle cliente-producto-color-semana": ["cod_cliente", "producto", "color", "week_start"],
    "Mercado-semana": ["mercado_cluster", "week_start"],
}.items():
    available = [key for key in keys if key in used_test.columns]
    if available:
        print(label, f"WAPE={wape(used_test, available):.2%}")

## 4. Predictores e importancia del boosting

El dashboard muestra que senales utiliza el boosting y como cambian sus
importancias por mercado. Las familias esperadas incluyen:

- Historico reciente de compra y volumen.
- Estacionalidad semanal/anual, incluyendo fase de temporada floral
  (`preparacion`, `pico` y `post-fiesta`) y distancia al pico.
- Indice estacional de mercado-producto-color-semana, que permite aprender
  la caida recurrente despues de San Valentin, Madres o fin de ano.
- Cliente, mercado, pais, producto y color.
- Frecuencia, tendencia y persistencia de compra.
- Predictores operativos disponibles para describir la serie.

La importancia ayuda a interpretar el modelo, pero no implica que una variable
cause la compra.

In [ ]:
predictors = read_forecast_output("solid_forecast_predictors")
importance = read_forecast_output("solid_forecast_feature_importance")
market_importance = read_forecast_output("solid_forecast_market_feature_importance")

display(predictors.head(30))
display(market_importance.sort_values("Importancia", ascending=False).head(30))

## 5. Calibracion por mercado y limitaciones

El modulo revisa si el modelo subpronostica de forma consistente un mercado en dos
mitades del backtest. Si hay evidencia sostenida, aplica una calibracion acotada
al nivel futuro del mercado. El boosting recibe fases de temporada para distinguir
preparacion, pico y salida posterior; tambien recibe un indice estacional semanal
por mercado-producto-color calculado solo con historia previa. En preparacion o
pico floral (por ejemplo San Valentin o Madres), puede reforzar el forecast con
la misma semana del ano anterior ajustada por la tendencia del mercado conocida
antes del ano pronosticado. La salida post-fiesta no se refuerza: su caida debe
ser aprendida por el modelo. Esta regla se reaplica al incorporar 2025 o 2026.

Limitaciones:

- El forecast estima demanda, no inventario disponible.
- No predice grado ni caja como objetivo.
- Un cambio comercial no observado historicamente puede invalidar el patron.
- Estructuras mixtas (`SURTIDO`, `SURTIDO_M`, `BOUQUET`, `BQT`, `COMBO`, `RAINBOW`) requieren un modelo
  posterior especifico de receta/composicion; no deben interpretarse desde este
  forecast de solidos.

In [ ]:
calibration = read_forecast_output("solid_forecast_market_calibration")
calibration

## 6. Validacion retrospectiva de temporadas

La salida `solid_forecast_historical_validation.csv` permite responder: que habria
pronosticado la logica estacional para semanas que ya ocurrieron. Esta salida
no reentrena el boosting en cada fecha historica: valida la referencia anual
ajustada sin usar informacion posterior al periodo evaluado.

La unidad correcta de prueba es una ventana consistente con la decision: 2,
5 u 8 semanas. El dashboard solo habilita anos e inicios cuya ventana esta
completa y tiene ano anterior comparable; WAPE y bias se calculan exclusivamente
en esa ventana y permite comparar las tres duraciones desde el mismo inicio.

In [ ]:
retrospective = read_forecast_output("solid_forecast_historical_validation", parse_dates=["week_start"])
year = int(retrospective["anio"].max())
start_week = 1  # Cambiar por una ventana de interes, por ejemplo semana 18 para Madres.
window_weeks = 5  # Alternativas visibles en el dashboard: 2, 5 u 8.
window = retrospective[
    retrospective["anio"].eq(year)
    & retrospective["semana_iso"].between(start_week, start_week + window_weeks - 1)
].copy()
summary = window.groupby("mercado_cluster", as_index=False).agg(
    real=("tallos", "sum"), predicho=("prediccion", "sum"), error_abs=("error_abs", "sum")
)
summary["WAPE"] = summary["error_abs"] / summary["real"].replace(0, float("nan"))
summary["Bias"] = (summary["predicho"] - summary["real"]) / summary["real"].replace(0, float("nan"))
summary

## 7. Visualizacion historica y forecast

La lectura recomendada en el dashboard es seleccionar mercado, cliente, producto y
color, comparar las lineas semanales por ano y superponer la linea de forecast.
Esto permite validar si la proyeccion mantiene una estacionalidad comercial
plausible para la seleccion.

In [ ]:
# Filtros para exploracion; usa listas vacias para no restringir.
MARKETS = []
CLIENTS = []
PRODUCTS = []
COLORS = []

view = future.copy()
for col, values in [
    ("mercado_cluster", MARKETS), ("cod_cliente", CLIENTS),
    ("producto", PRODUCTS), ("color", COLORS),
]:
    if values and col in view.columns:
        view = view[view[col].astype(str).isin([str(v) for v in values])]

plot_data = view.groupby("week_start", as_index=False)["tallos_estimados"].sum()
px.line(plot_data, x="week_start", y="tallos_estimados", title="Forecast SOLIDO para filtros seleccionados")

## 8. Ejecucion oficial en Bash

```bash
python run_forecast_solidos.py \
  --raw-historico "bases de datos historicas/historic_sales_acum.csv" \
  --output "resultados/forecast_solidos" \
  --test-weeks 8 \
  --horizon-weeks 8 \
  --no-cache
```

Cuando cambia la base acumulada o la clasificacion de pedidos, se debe volver a
ejecutar este modulo para que el dashboard cargue un modelo coherente con la fuente
actualizada.